# **Initialize the Model (a100)**

In [ ]:
import sys

# Uninstall potentially conflicting packages to ensure a clean slate
!pip uninstall -y vllm triton torch torchvision torchaudio Pillow tensorflow tensorflow-io

# Install torch and torchvision specifically for CUDA 12.x (common in Colab)
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121 --quiet

# Install vllm allowing it to pick a compatible version, with transformers pinned to 4.44.2
!pip install vllm==0.6.0 transformers==4.44.2 --quiet

Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 136.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 62.8 MB/s eta 0:00:00
 

In [ ]:
import os

def get_secret(key_name):
    """Fetch a secret from Colab, Kaggle, or env — whichever is available."""
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except Exception:
        pass
    return os.getenv(key_name)

hf_token = get_secret("HF_TOKEN")
assert hf_token, "HF_TOKEN not found — add it in Kaggle > Add-ons > Secrets"

# Log in so vLLM can pull the gated model
from huggingface_hub import login
login(token=hf_token)

# **NEW Humanized Prompts**

In [ ]:
import os
import re
import json
import math
import random
import unicodedata
from collections import defaultdict
from datasets import Dataset
from datasets import load_from_disk, Dataset
import json
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"
llm = LLM(
    model=MODEL_ID,
    quantization="awq_marlin",
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    tensor_parallel_size=1,
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 06-23 15:48:12 awq_marlin.py:89] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 06-23 15:48:12 llm_engine.py:213] Initializing an LLM engine (v0.6.0) with config: model='Qwen/Qwen2.5-32B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-32B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq_marlin, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_mod

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


INFO 06-23 15:49:12 model_runner.py:926] Loading model weights took 18.1477 GB
INFO 06-23 15:49:15 gpu_executor.py:122] # GPU blocks: 12964, # CPU blocks: 1024
INFO 06-23 15:49:17 model_runner.py:1217] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 06-23 15:49:17 model_runner.py:1221] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 06-23 15:49:47 model_runner.py:1335] Graph capturing finished in 30 secs.


In [ ]:
"""
generate_queries_v2.py
=======================
Replacement for the template-expansion pipeline. Instead of generating a few
templates and mechanically fanning them out over 23 indices / 3 modality phrases
(which makes a 1B model overfit to a fixed sentence shape), this asks the teacher
model to WRITE each query whole, like a real person. Variety comes from:

    who is asking  (persona)  x  what they want  (intent)  x  a rotating angle (flavor)

No placeholders, no post-hoc expansion. Single-sensor / single-modality queries
still occur, but as a small, naturally-phrased slice — matching how people
actually talk to a threat system — not 75% of the data.

Everything is grounded ONLY in values the model would see at inference:
  - fusion_confidence  (system-wide P(threat); a model output, NOT ground truth)
  - per sensor 0-22:   RF P(threat),  audio {mambo, bebop, background},  visual P(drone)
  - the static sensor -> direction/quadrant/hemisphere map
  - timestamp
No ground truth (drone_pos, is_threat_gt, nearest_sensor, triggered flags) and no
tool calls (no "show me the video / play the audio / pull the spectrogram").

Pipeline:  plan briefs by target distribution -> generate (batched vLLM)
           -> parse JSON -> tag -> dedup -> trim to distribution -> push to HF.

Run inside your notebook after `llm` and `tokenizer` are loaded, or `python generate_queries_v2.py`.
"""

import os
import re
import json
import math
import random
import unicodedata
from collections import defaultdict
from datasets import Dataset

random.seed(0)

# ============================================================================
# KNOBS
# ============================================================================
TOTAL_QUERIES      = 10_000     # set to 5000 if that's all you want
QUERIES_PER_BRIEF  = 12         # how many queries we ask for per model call
DEDUP_SURVIVAL     = 0.70       # fraction surviving dedup + anchor filter; used to size #briefs
BATCH_SIZE         = 16         # vLLM micro-batch
HF_USERNAME        = "JamesResearch1216"
HF_REPO_NAME       = "threat-detection-queries-v3"
PUSH_PRIVATE       = False

# ============================================================================
# SPATIAL VOCABULARY  (how people refer to regions; model phrases per persona)
# Index groupings are yours, verbatim — used later by the response labeler, and
# here only so technical personas can name regions correctly.
# ============================================================================
GROUP_INDICES = {
    "first quadrant":      list(range(11, 17)),
    "second quadrant":     list(range(5, 12)),
    "third quadrant":      list(range(0, 6)),
    "fourth quadrant":     list(range(16, 23)),
    "northern hemisphere": list(range(5, 17)),
    "southern hemisphere": list(range(0, 6)) + list(range(16, 23)),
    "eastern hemisphere":  list(range(11, 23)),
    "western hemisphere":  list(range(0, 12)),
}
REGION_SEEDS = [
    "the north side", "the south side", "the east side", "the west side",
    "the northeast", "the northwest", "the southeast", "the southwest",
    "the first quadrant", "the second quadrant",
    "the third quadrant", "the fourth quadrant",
]
DRONE_TYPES = ["mambo", "bebop"]

# Words that anchor a query to the drone / threat domain. A query missing all
# of these is too generic to use ("we good?", "narrow it down"). Lay terms
# included so civilian voices pass too.
ANCHOR_REGEX = re.compile(
    r"\b("
    r"drone|drones|uav|uavs|quadcopter|quadcopters|"
    r"threat|threats|hostile|hostiles|intruder|intruders|"
    r"sensor|sensors|detector|detectors|"
    r"detect|detects|detected|detecting|detection|detections|"
    r"signal|signals|transmission|transmissions|emission|emissions|"
    r"alert|alerts|alarm|alarms|warning|warnings|"
    r"contact|contacts|bogey|bogeys|bogie|bogies|"
    r"airspace|perimeter|"
    r"fly|flies|flying|aerial|overhead|"
    r"mambo|bebop|"
    r"rf|radio|audio|sonic|acoustic|sound|camera|cameras|"
    r"video|optical|visual|spectrogram"
    r")\b",
    re.IGNORECASE,
)


def has_anchor(q):
    return bool(ANCHOR_REGEX.search(q))

# ============================================================================
# PERSONAS  (5, with real tonal distance). `tech` = may use sensor numbers /
# light jargon. Examples are the strongest humanness signal — keep them real.
# ============================================================================
PERSONAS = {
    "formal_commander": dict(
        tech=True,
        style=("a military commander on the radio. Clipped, directive, proper "
               "register. Uses words like sector, status, confirm, report, "
               "contact, threat. Terse but grammatical. No slang, no emoji. "
               "Always names what is being asked about (drone, threat, contact)."),
        examples=[
            "Threat status, all sectors. Report.",
            "Confirm — are we tracking a drone to the north?",
            "I need a confidence assessment on the threat over the eastern perimeter.",
            "Is the drone detection RF-based or do we have optical confirmation?",
            "Single contact or multiple drones inbound? Give me a count.",
        ],
    ),
    "first_responder": dict(
        tech=True,
        style=("an on-scene responder, keyed up, focused on what to DO about "
               "the drone or threat and how close it is. Short. Drops "
               "capitalization sometimes, uses contractions. Not formal. "
               "Names the drone/threat explicitly — never just 'it' or 'this'."),
        examples=[
            "ok what's the drone situation here",
            "is the drone close to us or not",
            "do i need to move people or is this drone alert a false positive?",
            "which way is the drone coming from",
            "am i clear to stand down on this threat call yet",
        ],
    ),
    "neutral": dict(
        tech=True,
        style=("a normal person speaking plainly. Complete, polite sentences. "
               "No slang, little to no jargon. The default user. Always names "
               "the drone or threat explicitly in the question."),
        examples=[
            "Hi, is any drone activity being detected right now?",
            "Can you tell me where the drone seems to be located?",
            "How confident are you that this is a real drone threat?",
            "What kind of drone is it, if you can tell?",
            "Are there several drones out there or just one?",
        ],
    ),
    "casual": dict(
        tech=False,
        style=("a casual texter. mostly lowercase, occasional slang ('rn', "
               "'smth', 'nah'), contractions, light punctuation. relaxed but "
               "still clear about what they mean. avoid stacking too much slang "
               "in one message. ALWAYS names the drone or threat — never just "
               "'it' or 'this thing' without saying what it is."),
        examples=[
            "yo is there a drone or smth out there rn",
            "wait so is this drone threat actually dangerous or nah",
            "which side is the drone on, north or what",
            "are u sure its actually a drone or could it just be rf noise",
            "how many drones are we talking abt",
        ],
    ),
    "civilian": dict(
        tech=False,   # NEVER uses sensor numbers or jargon
        style=("a worried bystander with no technical knowledge. Plain, anxious. "
               "Uses lay words ('the drone', 'the flying thing', 'is it "
               "dangerous'). NEVER says sensor numbers, RF, modality, or "
               "confidence — they don't know those terms. ALWAYS names what "
               "they're worried about (the drone, the flying thing, the alert) "
               "explicitly in the question — never leaves it as a bare 'this' "
               "or 'it' or 'something happening'."),
        examples=[
            "is there a drone out there right now? should I be worried?",
            "are we safe from this drone?",
            "what is that flying thing — is it a drone?",
            "did a camera actually see a drone or are you just guessing?",
            "is the drone getting closer to us??",
        ],
    ),
}
ALL_PERSONAS = list(PERSONAS)
NON_CIVILIAN = ["formal_commander", "first_responder", "neutral", "casual"]
TECH_ONLY    = ["formal_commander", "first_responder", "neutral"]  # for number-heavy asks

# ============================================================================
# INTENTS  — weight = share of final set. `flavors` rotate the angle so repeated
# calls diverge. `target` tells the brief builder what concrete seed to inject.
# All are answerable from readings alone.
# ============================================================================
INTENTS = {
    "overall_presence": dict(
        weight=0.13, personas=ALL_PERSONAS, target="none",
        desc="whether there is a drone/threat anywhere right now, or an all-clear.",
        flavors=["a flat yes/no plus why", "an all-clear check", "first thing on shift",
                 "double-checking a hunch", "worried something was missed"],
    ),
    "localization_direction": dict(
        weight=0.11, personas=ALL_PERSONAS, target="region",
        desc="which direction / part of the field the activity is in.",
        flavors=["which way is it coming from", "narrow it to a side", "is it near a specific area",
                 "front or back", "is it moving toward us (answer from where signals are strongest)"],
    ),
    "severity_tasking": dict(
        weight=0.12, personas=ALL_PERSONAS, target="none",
        desc="how serious it is and what action to take (respond, dispatch, evacuate, hold).",
        flavors=["should I send someone", "do we evacuate", "how urgent on a gut level",
                 "is this worth scrambling for", "can we stand down", "what's the recommended move"],
    ),
    "situation_summary": dict(
        weight=0.10, personas=ALL_PERSONAS, target="none",
        desc="a quick overall picture / sitrep pulling everything together.",
        flavors=["give me the bottom line", "full sitrep", "what's going on right now",
                 "catch me up", "one-line summary"],
    ),
    "drone_identification": dict(
        weight=0.08, personas=ALL_PERSONAS, target="drone_type",
        desc="what kind of drone it is (the audio can tell mambo vs bebop).",
        flavors=["mambo or bebop", "is it a known model", "hobby drone or something bigger",
                 "what's it sound like it is", "can you ID it"],
    ),
    "region_threat": dict(
        weight=0.08, personas=ALL_PERSONAS, target="region",
        desc="threat status of a specific named area/region.",
        flavors=["status of one area", "is that side clear or active", "anything over there",
                 "compare two areas"],
    ),
    "count_active": dict(
        weight=0.05, personas=NON_CIVILIAN, target="none",
        desc="how many sensors are currently picking something up.",
        flavors=["how many detecting", "one contact or several", "widespread or isolated"],
    ),
    "crossmodal_corroboration": dict(
        weight=0.06, personas=NON_CIVILIAN, target="none",
        desc="whether multiple sensors or multiple detector types agree (raises trust).",
        flavors=["do the sensors agree", "is it confirmed by more than one thing",
                 "audio and camera both, or just one", "any sensor disagreeing"],
    ),
    "modality_driver": dict(
        weight=0.05, personas=ALL_PERSONAS, target="none",
        desc="which detector is driving the call — RF vs audio vs camera.",
        flavors=["is it RF or did a camera see it", "what tripped it",
                 "are you hearing it or seeing it", "is this just a radio signal"],
    ),
    "ranking_most_alarmed": dict(
        weight=0.05, personas=TECH_ONLY, target="none",
        desc="which sensor (or area) is most alarmed / strongest signal.",
        flavors=["which sensor is hottest", "where's the strongest signal",
                 "top two or three sensors", "loudest audio hit"],
    ),
    "confidence_explanation": dict(
        weight=0.06, personas=ALL_PERSONAS, target="none",
        desc="how sure the system is, and/or why it thinks what it thinks.",
        flavors=["how sure are you", "why do you think that", "what's this based on",
                 "could you be wrong", "talk me through it"],
    ),
    "single_sensor": dict(
        weight=0.06, personas=TECH_ONLY, target="sensor_index",
        desc="the status of one specific sensor.",
        flavors=["just that one sensor", "what's it seeing", "is it the one that's alarmed",
                 "quick read on it"],
    ),
    "single_modality_at_sensor": dict(
        weight=0.03, personas=TECH_ONLY, target="sensor_index",
        desc="one detector type at a specific sensor or area (RF, audio, or camera).",
        flavors=["just the camera there", "audio only", "what's the RF doing there"],
    ),
    "clear_regions": dict(
        weight=0.03, personas=ALL_PERSONAS, target="region",
        desc="which areas are quiet / confirmed clear of drones or threats.",
        flavors=["which side is clear of drones", "is that sector free of threats",
                 "anywhere I don't need to worry about a drone"],
    ),
}

# ============================================================================
# PROMPTS
# ============================================================================
SYSTEM_PROMPT = (
    "You write realistic questions that real people type to an airport "
    "counter-drone threat-detection system. You output ONLY the questions — never answers. "
    "The system can only reason from sensor readings (each of 23 sensors has a "
    "radio, an audio, and a camera detector) plus an overall threat estimate. So "
    "every question must be answerable from those readings. NEVER write questions "
    "that ask to see video, play audio, pull up images/spectrograms, or take any "
    "action other than giving information or advice. NEVER invent specific numbers "
    "or readings in the question. Sound human, not like a survey.\n\n"
    "CRITICAL — every question must STAND ALONE. A reader seeing only that one "
    "question, with zero prior context, must immediately know it is about drones, "
    "threats, sensors, signals, or airspace safety. Always anchor the question "
    "with at least one explicit domain word: drone, threat, contact, signal, "
    "sensor, alert, warning, detection, intruder, airspace, perimeter, audio, "
    "RF, camera, mambo, bebop, or a lay equivalent ('the flying thing', 'is "
    "something out there'). Vague questions like 'we good?', 'narrow it to a "
    "side', 'really sure?', 'how many are we talking about?' are FORBIDDEN — "
    "they could be about anything. Pronouns ('it', 'this', 'them', 'that') must "
    "have a clear referent inside the same question — write 'is the drone close' "
    "not 'is it close'."
)

ANTI_ROBOTIC = """\
Make them sound like real, messy human messages:
- Vary length: some short, some a full sentence or two. No uniform length.
- Vary how each one opens. Do NOT start multiple questions the same way.
- Use contractions and sentence fragments where it fits the persona.
- Match the persona's punctuation/capitalization (casual = lowercase/loose; commander = clean).
- Do not number them, do not add quotes around them, do not explain them.

EVERY question must be self-contained and clearly about the drone/threat system:
- Each one must explicitly mention at least one of: drone, threat, contact, signal,
  sensor, audio, RF, camera, alert, warning, detection, intruder, airspace,
  perimeter, mambo, bebop — or a lay equivalent ('the flying thing', 'is something
  flying out there'). Civilians use the lay versions; technical personas use the
  precise ones.
- A reader seeing ONLY this one question must instantly know it is about drone
  threat detection. Bad examples (do NOT write things like this): "we good or
  multiple spots?", "narrow it down to a side", "really sure about this?",
  "is it a bigger model not a small toy?", "east side stressing me out".
  Good versions of the same intent: "are drones being picked up in multiple
  spots?", "which side is the drone on?", "are you sure this drone reading
  is real?", "is the drone a larger model or just a hobby quad?", "is there
  a drone on the east side?".
- Pronouns ('it', 'this', 'them', 'that') must have a referent inside the same
  question. Write "is the drone close?" not "is it close?"."""


def _target_clause(intent_key, persona_key):
    """Inject a concrete, rotating seed so calls diverge — phrased by the model, not templated."""
    kind = INTENTS[intent_key]["target"]
    tech = PERSONAS[persona_key]["tech"]
    if kind == "region":
        regions = random.sample(REGION_SEEDS, k=min(4, len(REGION_SEEDS)))
        return (f"Anchor the questions around different areas (vary them): {', '.join(regions)}. "
                f"Refer to areas the way this persona would.")
    if kind == "sensor_index" and tech:
        idxs = sorted(random.sample(range(23), k=4))
        return (f"Refer to specific sensors by number; VARY which sensor across the questions "
                f"(e.g. some of {idxs}, but mix it up). A few may refer to an area instead of a number.")
    if kind == "drone_type":
        return ("Some should ask in general ('what kind of drone'), some by name "
                f"({' / '.join(DRONE_TYPES)}). Civilians ask in plain words ('is it a big one?').")
    return ""


def build_brief(intent_key, persona_key):
    intent = INTENTS[intent_key]
    persona = PERSONAS[persona_key]
    flavors = random.sample(intent["flavors"], k=min(2, len(intent["flavors"])))
    # rotate which persona examples are shown, so the model doesn't lock onto them
    ex = random.sample(persona["examples"], k=min(3, len(persona["examples"])))
    ex_block = "\n".join(f"  - {e}" for e in ex)
    target_clause = _target_clause(intent_key, persona_key)

    user = f"""Write {QUERIES_PER_BRIEF} different questions for this situation.

WHO IS ASKING: {persona['style']}

WHAT THEY WANT TO KNOW: {intent['desc']}
Angle(s) to lean into this batch: {', '.join(flavors)}.
{target_clause}

{ANTI_ROBOTIC}

Examples of how THIS persona sounds (match the voice, don't copy the content):
{ex_block}

Return a JSON array of exactly {QUERIES_PER_BRIEF} strings, nothing else."""
    return SYSTEM_PROMPT, user


# ============================================================================
# PLAN: how many briefs per (intent, persona) to hit the target distribution
# ============================================================================
def plan_briefs():
    total_w = sum(i["weight"] for i in INTENTS.values())
    plan, targets = [], {}
    for ik, intent in INTENTS.items():
        target_n = round(TOTAL_QUERIES * intent["weight"] / total_w)
        targets[ik] = target_n
        n_briefs = max(1, math.ceil(target_n / (QUERIES_PER_BRIEF * DEDUP_SURVIVAL)))
        personas = intent["personas"]
        for b in range(n_briefs):
            persona_key = personas[b % len(personas)]   # round-robin = even persona coverage
            plan.append((ik, persona_key))
    random.shuffle(plan)
    return plan, targets


# ============================================================================
# GENERATION (batched). `generate_fn` is injectable so the pipeline is testable
# without a GPU. Default wraps vLLM.
# ============================================================================
def vllm_generate_fn(briefs, llm, tokenizer, sampling_params):
    from vllm import SamplingParams  # noqa
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": sys}, {"role": "user", "content": usr}],
            tokenize=False, add_generation_prompt=True,
        )
        for (sys, usr) in briefs
    ]
    raw = []
    for start in range(0, len(prompts), BATCH_SIZE):
        chunk = prompts[start:start + BATCH_SIZE]
        print(f"  batch {start // BATCH_SIZE + 1}/{math.ceil(len(prompts)/BATCH_SIZE)}", end="\r", flush=True)
        for out in llm.generate(chunk, sampling_params):
            raw.append(out.outputs[0].text)
    print()
    return raw


# ============================================================================
# PARSE + TAG
# ============================================================================
def _coerce_json_array(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n?|\n?```$", "", text).strip()
    try:
        val = json.loads(text)
        if isinstance(val, list):
            return [str(x).strip() for x in val if str(x).strip()]
    except json.JSONDecodeError:
        pass
    m = re.search(r"\[.*\]", text, re.DOTALL)        # grab the first [...] blob
    if m:
        try:
            val = json.loads(m.group(0))
            if isinstance(val, list):
                return [str(x).strip() for x in val if str(x).strip()]
        except json.JSONDecodeError:
            pass
    # last resort: one question per line
    lines = [re.sub(r'^[\s\-\*\d\.\)"]+', "", ln).strip().strip('"')
             for ln in text.splitlines() if ln.strip()]
    return [ln for ln in lines if len(ln) > 2]


def parse_and_tag(plan, raw_texts):
    rows, n_bad = [], 0
    for (intent_key, persona_key), text in zip(plan, raw_texts):
        qs = _coerce_json_array(text)
        if not qs:
            n_bad += 1
            continue
        for q in qs:
            rows.append({"query": q, "intent": intent_key, "persona": persona_key})
    if n_bad:
        print(f"  ⚠️  {n_bad} briefs produced nothing parseable")
    return rows


# ============================================================================
# DEDUP + TRIM TO DISTRIBUTION
# ============================================================================
def _norm(q):
    q = unicodedata.normalize("NFKC", q).lower()
    q = re.sub(r"[^\w\s]", "", q)        # drop punctuation
    q = re.sub(r"\s+", " ", q).strip()
    return q


def dedup_and_trim(rows, targets):
    seen, deduped = set(), []
    no_anchor = 0
    for r in rows:
        k = _norm(r["query"])
        if not k or k in seen:
            continue
        if not has_anchor(r["query"]):
            no_anchor += 1
            continue
        seen.add(k)
        deduped.append(r)
    if no_anchor:
        print(f"  filtered {no_anchor} queries lacking a drone/threat anchor")

    # trim each intent down to its target so the final distribution is what we planned
    by_intent = defaultdict(list)
    for r in deduped:
        by_intent[r["intent"]].append(r)
    final = []
    for ik, items in by_intent.items():
        random.shuffle(items)
        keep = items[: targets.get(ik, len(items))]
        final.extend(keep)
        if len(items) < targets.get(ik, 0):
            print(f"  note: {ik} short — wanted {targets[ik]}, have {len(items)}")
    random.shuffle(final)
    return final


# ============================================================================
# DRIVER
# ============================================================================
def run(llm=None, tokenizer=None, hf_token=None, generate_fn=None):
    plan, targets = plan_briefs()
    briefs = [build_brief(ik, pk) for (ik, pk) in plan]
    print(f"Planned {len(briefs)} briefs -> targeting ~{sum(targets.values()):,} queries "
          f"across {len(INTENTS)} intents.")

    if generate_fn is None:
        from vllm import SamplingParams
        sampling_params = SamplingParams(temperature=1.05, top_p=0.95,
                                         max_tokens=1024, frequency_penalty=0.3)
        raw = vllm_generate_fn(briefs, llm, tokenizer, sampling_params)
    else:
        raw = generate_fn(briefs)

    rows = parse_and_tag(plan, raw)
    print(f"Parsed {len(rows):,} raw queries.")
    final = dedup_and_trim(rows, targets)
    print(f"Final after dedup + trim: {len(final):,} queries.")

    ds = Dataset.from_list(final)
    ds.save_to_disk("./queries_v2_ds")
    ds.to_json("./queries_v2.jsonl")

    if hf_token:
        repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
        print(f"Pushing to {repo_id} ...")
        ds.push_to_hub(repo_id, token=hf_token, private=PUSH_PRIVATE)
        print(f"✓ https://huggingface.co/datasets/{repo_id}")

    # report
    print("\nDistribution (intent):")
    counts = defaultdict(int)
    for r in final:
        counts[r["intent"]] += 1
    for ik in INTENTS:
        print(f"  {ik:<28} {counts[ik]:>5}  ({counts[ik]/max(1,len(final)):.1%})")
    print("\nDistribution (persona):")
    pc = defaultdict(int)
    for r in final:
        pc[r["persona"]] += 1
    for pk in PERSONAS:
        print(f"  {pk:<20} {pc[pk]:>5}  ({pc[pk]/max(1,len(final)):.1%})")
    print("\nSamples:")
    for r in random.sample(final, k=min(12, len(final))):
        print(f"  [{r['persona']:<16}|{r['intent']:<24}] {r['query']}")
    return ds


if __name__ == "__main__":
    # In your notebook you already have `llm`, `tokenizer`, and `hf_token`.
    try:
        run(llm=llm, tokenizer=tokenizer, hf_token=hf_token)          # noqa: F821
    except NameError:
        print("Load `llm`, `tokenizer`, `hf_token` first (your existing cells), then call run(...).")

Planned 1196 briefs -> targeting ~9,999 queries across 14 intents.


Processed prompts: 100%|██████████| 16/16 [00:10<00:00,  1.55it/s, est. speed input: 1392.58 toks/s, output: 285.13 toks/s]


Processed prompts: 100%|██████████| 16/16 [00:10<00:00,  1.49it/s, est. speed input: 1325.09 toks/s, output: 281.12 toks/s]


Processed prompts: 100%|██████████| 16/16 [00:10<00:00,  1.57it/s, est. speed input: 1418.80 toks/s, output: 276.67 toks/s]


Processed prompts: 100%|██████████| 16/16 [00:10<00:00,  1.57it/s, est. speed input: 1389.90 toks/s, output: 287.59 toks/s]

...

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  321kB /  321kB            

✓ https://huggingface.co/datasets/JamesResearch1216/threat-detection-queries-v3

Distribution (intent):
  overall_presence              1287  (12.9%)
  localization_direction        1089  (10.9%)
  severity_tasking              1188  (11.9%)
  situation_summary              990  (9.9%)
  drone_identification           792  (7.9%)
  region_threat                  792  (7.9%)
  count_active                   495  (5.0%)
  crossmodal_corroboration       594  (5.9%)
  modality_driver                495  (5.0%)
  ranking_most_alarmed           495  (5.0%)
  confidence_explanation         594  (5.9%)
  single_sensor                  594  (5.9%)
  single_modality_at_sensor      297  (3.0%)
  clear_regions                  297  (3.0%)

Distribution (persona):
  formal_commander      2356  (23.6%)
  first_responder       2299  (23.0%)
  neutral               2185  (21.9%)
  casual                1738  (17.4%)
  civilian              1421  (14.2%)

Samples:
  [first_responder |overall_presence  